<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260409_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
%%bash
node -v
npm -v

v20.19.0
10.8.2


In [17]:
%%bash
cd /content
npx create-vite@latest bootcamp --template react

└  Operation cancelled



In [18]:
%%bash
cd /content/bootcamp
npm install


added 2 packages, and audited 154 packages in 2s

37 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities


In [19]:
%%writefile /content/bootcamp/vite.config.js

import { defineConfig } from 'vite'
import react from '@vitejs/plugin-react'

export default defineConfig({
  plugins: [react()],
  server: {
    host: true,
    allowedHosts: [
      '.trycloudflare.com'
    ]
  }
})

Overwriting /content/bootcamp/vite.config.js


In [20]:
%%bash
cd /content/bootcamp
nohup npm run dev -- --host 0.0.0.0 > vite.log 2>&1 &
sleep 5
cat vite.log


> bootcamp@0.0.0 dev
> vite --host 0.0.0.0

2:07:40 AM [vite] (client) Re-optimizing dependencies because lockfile has changed

  VITE v8.0.7  ready in 341 ms

  ➜  Local:   http://localhost:5173/
  ➜  Network: http://172.28.0.12:5173/


In [21]:
%%bash
npm install -g cloudflared


changed 1 package in 2s


In [22]:
%%bash
nohup cloudflared tunnel --url http://localhost:5173 > tunnel.log 2>&1 &
sleep 8
grep -o 'https://.*\.trycloudflare\.com' tunnel.log

https://roger-distinction-keyword-provincial.trycloudflare.com




---



In [23]:
%%writefile /content/bootcamp/src/useLocalStorage.js

import { useState, useEffect } from 'react';

function useLocalStorage(key, initialValue) {
  const [value, setValue] = useState([]);

  useEffect(function () {
    const saved = localStorage.getItem(key);
    if (saved) {
      setValue(JSON.parse(saved));
    } else {
      setValue(initialValue);
    }
  }, []);

  useEffect(function () {
    localStorage.setItem(key, JSON.stringify(value));
  }, [value]);

  return [value, setValue];
}

export default useLocalStorage;

Overwriting /content/bootcamp/src/useLocalStorage.js


In [24]:
%%writefile /content/bootcamp/src/useTodos.js

import useLocalStorage from './useLocalStorage';

function useTodos() {
  const [todos, setTodos] = useLocalStorage('todo_data', []);

  function addTodo(text) {
    if (text.trim() === '') {
      return;
    }
    const newTodo = {
      id: Date.now(),
      title: text,
      done: false,
    };
    setTodos([...todos, newTodo]);
  }

  function toggleTodo(id) {
    const newTodos = todos.map(function (todo) {
      if (todo.id === id) {
        return { ...todo, done: !todo.done };
      }
      return todo;
    });
    setTodos(newTodos);
  }

  function removeTodo(id) {
    const newTodos = todos.filter(function (todo) {
      return todo.id !== id;
    });
    setTodos(newTodos);
  }

  return {
    todos,
    addTodo,
    toggleTodo,
    removeTodo,
  };
}

export default useTodos;

Overwriting /content/bootcamp/src/useTodos.js


In [25]:
%%writefile /content/bootcamp/src/TodoForm.jsx

function TodoForm(props) {
  return (
    <div style={styles.formBox}>
      <input
        type="text"
        value={props.text}
        onChange={function (e) {
          props.setText(e.target.value);
        }}
        placeholder="할 일을 입력하세요"
        style={styles.input}
      />
      <button onClick={props.addTodo} style={styles.button}>
        추가
      </button>
    </div>
  );
}

const styles = {
  formBox: {
    display: 'flex',
    gap: '8px',
    marginBottom: '20px',
  },
  input: {
    flex: 1,
    padding: '10px',
    border: '1px solid #ccc',
    borderRadius: '8px',
  },
  button: {
    padding: '10px 14px',
    border: 'none',
    borderRadius: '8px',
    backgroundColor: '#222',
    color: '#fff',
    cursor: 'pointer',
  },
};

export default TodoForm;

Overwriting /content/bootcamp/src/TodoForm.jsx


In [26]:
%%writefile /content/bootcamp/src/TodoList.jsx

function TodoList(props) {
  return (
    <div>
      {props.todos.map(function (todo) {
        return (
          <div key={todo.id} style={styles.item}>
            <div style={styles.leftBox}>
              <input
                type="checkbox"
                checked={todo.done}
                onChange={function () {
                  props.toggleTodo(todo.id);
                }}
              />
              <span
                style={{
                  ...styles.text,
                  textDecoration: todo.done ? 'line-through' : 'none',
                  color: todo.done ? '#999' : '#222',
                }}
              >
                {todo.title}
              </span>
            </div>
            <button
              onClick={function () {
                props.removeTodo(todo.id);
              }}
              style={styles.deleteButton}
            >
              삭제
            </button>
          </div>
        );
      })}
    </div>
  );
}

const styles = {
  item: {
    display: 'flex',
    justifyContent: 'space-between',
    alignItems: 'center',
    padding: '12px 0',
    borderBottom: '1px solid #eee',
  },
  leftBox: {
    display: 'flex',
    alignItems: 'center',
    gap: '10px',
  },
  text: {
    fontSize: '16px',
  },
  deleteButton: {
    padding: '6px 10px',
    border: 'none',
    borderRadius: '6px',
    backgroundColor: '#d93025',
    color: '#fff',
    cursor: 'pointer',
  },
};

export default TodoList;

Overwriting /content/bootcamp/src/TodoList.jsx


In [27]:
%%writefile /content/bootcamp/src/App.jsx

import { useState } from 'react';
import TodoForm from './TodoForm';
import TodoList from './TodoList';
import useTodos from './useTodos';

function App() {
  const [text, setText] = useState('');

  const {
    todos,
    addTodo,
    toggleTodo,
    removeTodo,
  } = useTodos();

  const remainCount = todos.filter(function (todo) {
    return todo.done === false;
  }).length;

  return (
    <div style={styles.wrap}>
      <h1 style={styles.title}>Todo List</h1>
      <p style={styles.desc}>남은 할 일: {remainCount}개</p>

      <TodoForm
        text={text}
        setText={setText}
        addTodo={function () {
          addTodo(text);
          setText('');
        }}
      />

      <TodoList
        todos={todos}
        toggleTodo={toggleTodo}
        removeTodo={removeTodo}
      />
    </div>
  );
}

const styles = {
  wrap: {
    width: '500px',
    margin: '40px auto',
    padding: '24px',
    border: '1px solid #ddd',
    borderRadius: '12px',
    backgroundColor: '#fafafa',
  },
  title: {
    marginBottom: '10px',
  },
  desc: {
    marginBottom: '20px',
    color: '#555',
  },
};

export default App;

Overwriting /content/bootcamp/src/App.jsx




---



In [28]:
%%bash
cat /content/bootcamp/vite.log


> bootcamp@0.0.0 dev
> vite --host 0.0.0.0

2:07:40 AM [vite] (client) Re-optimizing dependencies because lockfile has changed

  VITE v8.0.7  ready in 341 ms

  ➜  Local:   http://localhost:5173/
  ➜  Network: http://172.28.0.12:5173/


In [29]:
%%bash
cat tunnel.log

2026-04-09T02:07:47Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-09T02:07:47Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-09T02:07:50Z INF +--------------------------------------------------------------------------------------------+
2026-04-09T02:07:50Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-09T02:07:50Z INF |  https://roger-distinction-keyword-provincial.trycloud